In [1]:
import sys
!{sys.executable} -m pip install aquacrop
!{sys.executable} -m pip install stable-baselines3[extra]


  Using cached stable_baselines3-2.9.0-py3-none-any.whl.metadata (5.0 kB)
  Using cached torch-2.12.1-cp312-cp312-win_amd64.whl.metadata (31 kB)
  Using cached tensorboard-2.20.0-py3-none-any.whl.metadata (1.8 kB)
Using cached stable_baselines3-2.9.0-py3-none-any.whl (187 kB)
Using cached torch-2.12.1-cp312-cp312-win_amd64.whl (123.0 MB)
Using cached tensorboard-2.20.0-py3-none-any.whl (5.5 MB)

   ---------------------------------------- 0/3 [torch]
   ---------------------------------------- 0/3 [torch]
   ---------------------------------------- 0/3 [torch]
   ---------------------------------------- 0/3 [torch]
   ---------------------------------------- 0/3 [torch]
   ---------------------------------------- 0/3 [torch]
   ---------------------------------------- 0/3 [torch]
   ---------------------------------------- 0/3 [torch]
   ---------------------------------------- 0/3 [torch]
   ---------------------------------------- 0/3 [torch]
   --------------------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [12]:
import os
import gymnasium as gym
from stable_baselines3 import PPO 
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy
from gymnasium import Env
from gymnasium import Discrete, Box
import numpy as np
import random

class AquaCropEnv(Env):
    def __init__(self):
        self.action_space = self.action_space = Box(low=np.array([0], dtype=np.float32),
                                                    high=np.array([999]),dtype=np.float32)
        self.observation_space = Box(low=[0, 0, -20, 0, 0], 
                                     high=[100, 50, 60, 100, 100000])
   
        self.season_length = 30
        self.day = 0
        self.state = None
    def step(self, action):
        pass
    def render(self):
        pass
    def reset(self):
        pass
        


In [55]:
import pandas as pd
from aquacrop import AquaCropModel, Soil, Crop, InitialWaterContent, IrrigationManagement
from aquacrop.utils import prepare_weather, get_filepath

weather_file_path = get_filepath('tunis_climate.txt')

# вручную задаём полив
irrigation_schedule = pd.DataFrame({
    'Date': pd.to_datetime([
        '1979-10-01',
        '1979-10-02',
        '1979-10-03',
        '1979-10-04',
        '1979-10-05',
    ]),
    'Depth': [100, 200, 0, 50, 100]   # мм воды
})

irrigation_management = IrrigationManagement(
    irrigation_method=3,
    Schedule=irrigation_schedule
)

model_os = AquaCropModel(
    sim_start_time="1979/10/01",
    sim_end_time="1980/05/30",
    weather_df=prepare_weather(weather_file_path),
    soil=Soil(soil_type='SandyLoam'),
    crop=Crop('Wheat', planting_date='10/01'),
    initial_water_content=InitialWaterContent(value=['FC']),
    irrigation_management=irrigation_management
)

model_os.run_model(till_termination=True)

storage = model_os.get_water_storage()
water = model_os.get_water_flux()

result = storage[['dap', 'th1', 'th2', 'th3', 'th4']].copy()
result['irrigation'] = water['IrrDay']

result.head(20)

,dap,th1,th2,th3,th4,irrigation
0,1.0,0.372600,0.280000,0.220000,0.220000,25.0
1,2.0,0.377000,0.285814,0.224446,0.224204,25.0
2,3.0,0.190743,0.222643,0.224371,0.224318,0.0
3,4.0,0.370400,0.250983,0.220393,0.220389,25.0
4,5.0,0.370400,0.286063,0.224501,0.223886,25.0
5,6.0,0.190040,0.223031,0.224424,0.224268,0.0
6,7.0,0.173493,0.220275,0.220398,0.220384,0.0
7,8.0,0.155211,0.220025,0.220037,0.220035,0.0
8,9.0,0.139316,0.220002,0.220003,0.220003,0.0
9,10.0,0.126390,0.220000,0.220000,0.220000,0.0
